# Tutorial de uso de la base ORM

Este notebook valida el flujo local completo de la capa ORM del IGSM:

- crear una base SQLite nueva en un directorio temporal;
- crear el esquema SQLAlchemy desde `database.models.Base`;
- sembrar datos de referencia con `database.seed`;
- leer y escribir mediante las funciones públicas de `database.repositories`.

La base temporal mantiene el ejemplo aislado de `database/igsm_dev.sqlite3` y de cualquier `DATABASE_URL` configurado para la aplicación. Las celdas imprimen resúmenes pequeños para facilitar la revisión, y los `assert` hacen fallar el notebook si el comportamiento esperado cambia.


## 1. Preparación

Ejecute este notebook desde la raíz del repositorio o desde la carpeta `tests`. La celda de preparación busca hacia arriba hasta encontrar la raíz del proyecto y la agrega a `sys.path` para poder importar los paquetes locales `database` y `data`.

Si aparece `ModuleNotFoundError: No module named 'sqlalchemy'`, instale primero las dependencias del proyecto en el entorno activo del kernel.


In [ ]:
from datetime import date
from pathlib import Path
from pprint import pprint
import sys
from uuid import uuid4

from sqlalchemy import func, inspect, select


def find_project_root(start: Path) -> Path:
    """Find the repository root from a starting directory.

    Args:
        start: Directory where the search starts.

    Returns:
        Repository root containing `database/` and `data/`.

    Raises:
        RuntimeError: If the repository root cannot be found.
    """

    for candidate in (start, *start.parents):
        if (candidate / "database").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("No se encontró la raíz del proyecto con database/ y data/.")


ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from database.models import (  # noqa: E402
    Base,
    DMAxis,
    DMIndicator,
    DMMunicipality,
    DMService,
    FactIndicatorResponse,
    FactMaturityThreshold,
    FactStageWeight,
)
from database.repositories import (  # noqa: E402
    get_indicators_for_service,
    get_latest_indicator_responses,
    get_latest_maturity_thresholds,
    get_latest_responses_for_municipality,
    get_latest_stage_weights,
    get_municipality_completion_statistics,
    get_national_statistics,
    get_services_for_municipality,
    list_municipalities,
    submit_indicator_responses,
)
from database.seed import seed_all  # noqa: E402
from database.session import get_engine, session_scope  # noqa: E402


TEMP_PARENT = ROOT / ".test_tmp"
TEMP_PARENT.mkdir(exist_ok=True)
DB_PATH = TEMP_PARENT / f"orm_usage_{uuid4().hex}.sqlite3"
DATABASE_URL = f"sqlite:///{DB_PATH.as_posix()}"
END_DATE = date(2025, 1, 1)
engine = get_engine(DATABASE_URL)

print(f"Raíz del proyecto: {ROOT}")
print(f"Base SQLite temporal: {DB_PATH}")


## 2. Crear el esquema

`Base.metadata.create_all(engine)` crea todas las tablas ORM en la base SQLite temporal. Es el mismo metadata que usa la capa de datos de la aplicación.


In [ ]:
Base.metadata.create_all(engine)

tables = sorted(inspect(engine).get_table_names())
print(f"Tablas creadas: {len(tables)}")
pprint(tables)

required_tables = {
    "dm_municipality",
    "dm_axis",
    "dm_service",
    "dm_indicator",
    "fact_indicator_response",
    "fact_stage_weight",
    "fact_maturity_threshold",
}
assert required_tables.issubset(tables)
assert not any(table.startswith("app_") for table in tables)


## 3. Sembrar datos de referencia

`seed_all(session=session)` carga municipalidades, ejes, servicios, indicadores, pesos de etapa y umbrales de madurez desde los datos fuente del proyecto. La celda siguiente cuenta filas clave como prueba rápida.


In [ ]:
models_to_count = {
    "municipalidades": DMMunicipality,
    "ejes": DMAxis,
    "servicios": DMService,
    "indicadores": DMIndicator,
    "pesos_etapa": FactStageWeight,
    "umbrales_madurez": FactMaturityThreshold,
}

with session_scope(DATABASE_URL) as session:
    seed_all(session=session)
    counts = {
        name: session.scalar(select(func.count()).select_from(model))
        for name, model in models_to_count.items()
    }

print("Conteos de referencia sembrados:")
pprint(counts)

assert counts["municipalidades"] > 0
assert counts["servicios"] > 0
assert counts["indicadores"] > 0
assert counts["pesos_etapa"] > 0
assert counts["umbrales_madurez"] > 0


## 4. Leer mediante la API del repositorio

Las funciones del repositorio devuelven listas y diccionarios pensados para la aplicación. Así la interfaz no necesita consultar SQLAlchemy directamente.


In [ ]:
with session_scope(DATABASE_URL) as session:
    municipalities = list_municipalities(session=session)
    sample_municipality = municipalities[0]
    services = get_services_for_municipality(sample_municipality["codigo"], session=session)
    first_service = next(iter(services.values()))
    indicators = get_indicators_for_service(first_service["service_code"], session=session)
    weights = get_latest_stage_weights(end_date=END_DATE, session=session)
    thresholds = get_latest_maturity_thresholds(end_date=END_DATE, session=session)

print("Municipalidad de ejemplo:")
pprint(sample_municipality)

print(f"\nServicios aplicables: {len(services)}")
pprint(list(services)[:5])

print(f"\nPrimer servicio: {first_service['name']} ({first_service['service_code']})")
print(f"Indicadores para este servicio: {len(indicators)}")
pprint(indicators[:3])

print("\nPesos de etapa vigentes:")
pprint(weights)

print("\nUmbrales de madurez vigentes:")
pprint(thresholds)

assert municipalities
assert services
assert indicators
assert abs(sum(weights.values()) - 1.0) < 0.000001


## 5. Escribir respuestas y leer completitud

Este flujo guarda un conjunto pequeño de respuestas numéricas en `fact_indicator_response` con `submit_indicator_responses`. Luego lee las respuestas más recientes a `END_DATE`, todas las respuestas más recientes nacionales y la completitud nacional/municipal desde la API pública del ORM.


In [ ]:
with session_scope(DATABASE_URL) as session:
    municipality = list_municipalities(session=session)[0]
    services = get_services_for_municipality(municipality["codigo"], session=session)

    responses = {}
    for service in services.values():
        for indicator in get_indicators_for_service(service["service_code"], session=session):
            responses[indicator["codigo"]] = 1.0
            if len(responses) >= 12:
                break
        if len(responses) >= 12:
            break

    submission = submit_indicator_responses(
        municipality["codigo"],
        END_DATE,
        responses,
        session=session,
    )

    latest = get_latest_responses_for_municipality(municipality["codigo"], end_date=END_DATE, session=session)
    all_latest = get_latest_indicator_responses(end_date=END_DATE, session=session)
    stats = get_national_statistics(end_date=END_DATE, session=session)
    municipality_stats = get_municipality_completion_statistics(
        municipality["codigo"],
        end_date=END_DATE,
        session=session,
    )
    fact_response_count = session.scalar(select(func.count()).select_from(FactIndicatorResponse))
    total_municipalities = session.scalar(select(func.count()).select_from(DMMunicipality))
    total_indicators = session.scalar(select(func.count()).select_from(DMIndicator))

print(f"Municipalidad enviada: {municipality['nombre']} ({municipality['codigo']})")
print(f"Hechos creados o actualizados: {len(submission['fact_response_ids'])}")
print(f"Respuestas recientes leídas: {len(latest)}")
print(f"Respuestas nacionales recientes: {len(all_latest)}")
print(f"Filas en fact_indicator_response: {fact_response_count}")

print("\nCompletitud nacional:")
pprint(stats)

print("\nCompletitud municipal:")
pprint(municipality_stats)

assert len(submission["fact_response_ids"]) == len(responses)
assert submission["end_date"] == END_DATE.isoformat()
assert submission["responses_count"] == len(responses)
assert "score_total" not in submission
assert "nivel" not in submission
assert "calculation" not in submission
assert latest
assert len(all_latest) == len(responses)
assert fact_response_count == len(submission["fact_response_ids"])
assert stats["respuestas_esperadas"] == total_municipalities * total_indicators
assert stats["respuestas_recibidas"] == len(responses)
assert stats["pct_completitud"] == round(len(responses) / (total_municipalities * total_indicators) * 100, 2)
assert municipality_stats["respuestas_esperadas"] == total_indicators
assert municipality_stats["respuestas_recibidas"] == len(responses)
assert municipality_stats["pct_completitud"] == round(len(responses) / total_indicators * 100, 2)


## 6. Limpieza

Cierre el engine SQLAlchemy antes de eliminar el directorio temporal. Esto ayuda especialmente en Windows, donde los archivos SQLite pueden quedar bloqueados si hay conexiones abiertas.


In [ ]:
engine.dispose()
if DB_PATH.exists():
    DB_PATH.unlink()

print(f"Engine cerrado y base temporal eliminada: {DB_PATH}")
